# Model based grading

A grader turns an output into a measurable score, usually 1 to 10. Three kinds:

| Grader | How | Good for |
|---|---|---|
| Code | your code checks the output | format, syntax, length, required words |
| Model | another Claude call grades it | quality, task following, completeness |
| Human | a person reads it | anything, but slow |

A model grader gets the task and the output. Ask for **strengths, weaknesses and reasoning**
before the score: without them models tend to give a middling ~6 to everything.

Note: the code on the lesson page uses a plain string with `{task}` / `{solution}`, so nothing
is filled in. The course notebook uses an f-string with XML tags, which is what is used here.

## Setup

The client reads `ANTHROPIC_API_KEY` (and `ANTHROPIC_BASE_URL`, if set) from `.env` in the repo root.

In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## Helpers

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    # The API rejects system=None, so optional params are only sent when set
    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return "".join(block.text for block in message.content if block.type == "text")

## The model grader

In [3]:
import json
from statistics import mean


def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
"""

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

## Eval pipeline with grading

In [4]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [5]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [6]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

## Run it

In [7]:
with open("dataset.json") as f:
    dataset = json.load(f)

In [8]:
results = run_eval(dataset)

for result in results:
    print(f"\n[{result['score']}] {result['test_case']['task']}")
    print(f"    {result['reasoning']}")

Average score: 7.333333333333333

[7] Write a Python function that extracts the AWS account ID from an ARN string
    The solution correctly solves the core task of extracting account IDs from ARNs. The logic is sound, documentation is thorough, and test coverage is good. However, the exception handling is too broad and masks original errors, the validation could be stricter (checking 12-digit length), and the behavior is somewhat inconsistent between services with no account ID vs. completely invalid ARNs. These are relatively minor issues that don't break functionality but could cause problems in production environments.

[7] Create a JSON object that represents an AWS IAM policy allowing read-only access to S3 bucket objects
    The solution provides a functional and well-structured IAM policy that correctly implements read-only S3 access. The separation of object-level and bucket-level permissions with proper ARN syntax is correct. However, the inclusion of versioning-related permi